In [1]:
import faiss

In [3]:
import numpy as np
import pandas as pd

In [4]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [7]:
import torch

torch.cuda.is_available()

True

In [8]:
!gdown https://drive.google.com/file/d/141BAMKzuPuwWcecaefevoSkziUrqkuO5/view?usp=sharing --fuzzy

Downloading...
From: https://drive.google.com/uc?id=141BAMKzuPuwWcecaefevoSkziUrqkuO5
To: /content/employees.parquet
100% 89.2M/89.2M [00:01<00:00, 53.7MB/s]


In [9]:
!gdown https://drive.google.com/file/d/1Qks6QsznD2JeFlg3VTw1PXNi20cpTvKf/view?usp=sharing --fuzzy

Downloading...
From: https://drive.google.com/uc?id=1Qks6QsznD2JeFlg3VTw1PXNi20cpTvKf
To: /content/orcs.parquet
100% 2.52M/2.52M [00:00<00:00, 14.9MB/s]


In [8]:
orcs = pd.read_parquet('orcs.parquet')

In [9]:
df = pd.read_parquet('employees.parquet')

In [10]:
df.head()

,name,surname,fathername,gender,birthdate,inn,passport
0,жумавой,волоколамц*в,садагет оглы,м,1973-12-24,804926960301,7880355409
1,мубн,невмятуллин,"али,вич",м,1984-07-06,111068355926,1411814346
2,"ан,рей",карпоэ,вяч.славовеч,м,1962-03-28,617713534516,3428889207
3,олрга,рубцова,викторовна,ж,1961-10-13,530330958056,5497629378
4,сергей,леусенко,григорьевич,м,1998-09-10,872242822954,8988513519


In [11]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from tqdm import tqdm


class RacistTransformer(TransformerMixin, BaseEstimator):
    def __init__(self):
        self.column_ngrams = {
            'name': (2, 4),
            'surname': (2, 4),
            'fathername': (2, 4),
            'inn': (3, 3),
            'passport': (3, 3),
            'birthdate': (3, 3)
        }
        self.vecs = {
        }

    def fit(self, X, y=None):
        X.fillna('', inplace=True)
        for col, ngram in tqdm(self.column_ngrams.items()):
            vec = Pipeline([
                ('vectorize', TfidfVectorizer(
                    ngram_range=ngram,
                    analyzer='char',
                    min_df=2,
                    max_features=100,
                    
                )),
            ])
            self.vecs[col] = vec.fit(X[col])
        return self

    def transform(self, X):
        X.fillna('', inplace=True)
        dfs = []
        for col in self.column_ngrams:
            df = pd.DataFrame.sparse.from_spmatrix(self.vecs[col].transform(X[col]))
            dfs.append(df)
        return pd.concat(dfs, axis='columns')

In [12]:
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA

In [13]:
dim = 50

In [14]:
model = Pipeline([
    ('vec', RacistTransformer()),
    ('compress', PCA(n_components=dim))
])

In [15]:
df_vec = model.fit_transform(df)

100%|██████████| 6/6 [00:39<00:00,  6.57s/it]


In [18]:
orcs_vec = model.transform(orcs)

In [19]:
df_vec = df_vec.astype(np.float32)
orcs_vec = orcs_vec.astype(np.float32)

In [20]:
df_vec.shape, orcs_vec.shape

((2011759, 50), (47633, 50))

In [21]:
n_centroids = int(np.sqrt(len(df_vec)))

In [22]:
index = faiss.index_factory(dim, f"IVF{n_centroids},Flat")

In [ ]:
res = faiss.StandardGpuResources() 
index = faiss.index_cpu_to_gpu(res, 0, index)

In [26]:
index.train(df_vec)

In [27]:
index.add(df_vec)

In [37]:
topn = 1
D, I = index.search(orcs_vec, topn)

In [38]:
I.shape

(47633, 1)

In [39]:
len(set(I[:, 0]))

44437

In [42]:
sub = pd.DataFrame({
    'id': np.arange(0, len(I), 1),
    'orig_index': I[:, 0]
})

sub.to_parquet('/content/sub.parquet')

In [47]:
from google.colab import files
files.download("/content/sub.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>